# PriceDekho - Smart Car Price Estimator using ML & Deep Learning
This notebook demonstrates the end-to-end training and inference pipeline for PriceDekho.

In [ ]:
import os

# Set logical directory to parent so we can access project files natively
if os.path.basename(os.getcwd()) == 'project':
    os.chdir('..')
print(f"Working dir: {os.getcwd()}")

### 1. Machine Learning Model
A Random Forest Regressor to predict the base price of the car from tabular data.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import joblib

# Load data
df = pd.read_csv('car data.csv')

# Encode categorical
le = LabelEncoder()
df['Fuel_Type'] = le.fit_transform(df['Fuel_Type'])
df['Seller_Type'] = le.fit_transform(df['Seller_Type'])
df['Transmission'] = le.fit_transform(df['Transmission'])

# Features & target
X = df.drop(['Selling_Price', 'Car_Name'], axis=1)
y = df['Selling_Price']

# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Save
joblib.dump(model, 'price_model.pkl')

print('✅ Price model trained & saved')

### 2. Deep Learning Model (YOLOv8)
Fine-tune a YOLO model to detect `scratch`, `dent`, and `broken` on car images.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='scratch-dent-car-3/data.yaml',
    epochs=20,   # keep low for laptop
    imgsz=640
)

### 3. Inference & Evaluation
Combine both ML and DL predictions to calculate final selling price with damage penalty.

In [ ]:
import joblib
from ultralytics import YOLO
import cv2
from visualise import detect_and_draw
import matplotlib.pyplot as plt

# Load models
price_model = joblib.load('price_model.pkl')
yolo_model = YOLO('runs/detect/train3/weights/best.pt')

def calculate_damage(results, image_path):
    weights = {'scratch':0.3, 'dent':0.6, 'broken':1.0}

    img = cv2.imread(image_path)
    h, w, _ = img.shape
    image_area = h * w

    total_score = 0

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            label = r.names[cls]
            x1, y1, x2, y2 = box.xyxy[0]
            box_area = (x2-x1)*(y2-y1)
            ratio = box_area / image_area
            total_score += ratio * weights[label]

    return total_score

def predict_all(features, image_path):
    base_price = float(price_model.predict([features])[0])
    img, detected, results = detect_and_draw(image_path)
    damage_score = float(calculate_damage(results, image_path))
    final_price = float(base_price * (1 - damage_score * 0.5))

    return img, {
        'base_price': base_price,
        'damage_types': detected,
        'damage_score': damage_score,
        'final_price': final_price
    }

# TEST RUN
img, result = predict_all(
    [2015, 5.0, 50000, 1, 0, 1, 0],
    'scratch-dent-car-3/test/images/122_jpg.rf.e35dd4aef55b20f4a997c8b8d62a4ae1.jpg'
)

print("Final Prediction Details:", result)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Final Output Window')
plt.show()